# 🚀 IGOTYOU — Custom YOLOv11 Instagram Profile & Grid Detector
โน้ตบุ๊กสำหรับสร้างชุดข้อมูลจำลองหน้าจอ Instagram (Synthetic Dataset) และเทรนโมเดล **YOLOv11-Nano** เพื่อนำไปรันบนเว็บ **IGOTYOU**

### 🎯 ความสามารถของโมเดล (5 Classes + Negative Samples):
- `0: post_tile` — รูปโพสต์จริงแต่ละช่อง (รองรับทั้ง 1-9 รูป, สัดส่วน 3:4 และ 1:1, ข้ามช่องว่างอัตโนมัติ)
- `1: avatar` — รูปโปรไฟล์วงกลมด้านบน
- `2: ig_stats_bar` — แถบตัวเลขสถิติ (โพสต์ / ผู้ติดตาม / กำลังติดตาม) ใช้ยืนยันว่าเป็นหน้าโปรไฟล์ IG จริง
- `3: ig_tab_bar` — แถบไอคอนสลับแท็บ (Grid / Reels / Tagged)
- `4: empty_or_private` — สัญลักษณ์กรณี **ไม่มีรูปโพสต์ (0 Posts)** หรือ **บัญชีส่วนตัว (Private Account)**
- **Negative Samples (20%)** — ภาพสลิปโอนเงิน, แชท, รูปถ่ายเดี่ยวเต็มจอ (สอนให้ AI ปฏิเสธภาพที่ไม่ใช่ IG ทันที)

---
**วิธีใช้งาน:**
1. ไปที่เมนูด้านบน `Runtime` -> `Change runtime type` -> เลือก **T4 GPU** (หรือ L4 / A100)
2. กด `Runtime` -> `Run all` (หรือกดรันทีละ Cell จากบนลงล่าง)
3. เมื่อเสร็จสิ้น Cell สุดท้าย เบราว์เซอร์จะดาวน์โหลดไฟล์ **`ig_detector.onnx`** ลงเครื่องอัตโนมัติ ให้นำไปวางที่ `public/models/ig_detector.onnx` ในโปรเจกต์ได้เลย!

In [ ]:
# @title 1. ติดตั้งไลบรารี Ultralytics YOLOv11 และ ONNX Runtime
!pip install -q ultralytics onnx onnxslim onnxruntime pillow numpy requests tqdm
import torch
print(f"✅ PyTorch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ คำเตือน: ยังไม่ได้เปิด GPU! แนะนำให้ไปที่ Runtime -> Change runtime type -> เลือก GPU ก่อนครับ")

In [ ]:
# @title 2. ดาวน์โหลดภาพถ่ายจริงหลากหลายสไตล์ (สำหรับนำมาสุ่มใส่ในช่องโพสต์และ Avatar)
import os
import io
import random
import requests
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

os.makedirs("raw_photos", exist_ok=True)

def fetch_photo(idx):
    path = f"raw_photos/photo_{idx:03d}.jpg"
    if os.path.exists(path):
        return path
    try:
        r = requests.get(f"https://picsum.photos/seed/igotyou_{idx}/400/500", timeout=8)
        if r.status_code == 200:
            img = Image.open(io.BytesIO(r.content)).convert("RGB")
            img.save(path, quality=88)
            return path
    except Exception:
        pass
    return None

print("📥 กำลังดาวน์โหลดภาพถ่ายตัวอย่าง 120 ภาพ...")
with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(tqdm(ex.map(fetch_photo, range(120)), total=120))

photo_paths = [p for p in results if p is not None]
print(f"✅ ดาวน์โหลดสำเร็จ {len(photo_paths)} ภาพ (พร้อมสร้างภาพ Procedural เสริมอัตโนมัติ)")

In [ ]:
# @title 3. สร้างชุดข้อมูล IG จำลอง 1,500 ภาพ (รวมเคส <6 รูป, 0 รูป/Private, แท็บเล็ต และภาพหลอกสลิป/แชท/มีม)
import shutil
import yaml

DATASET_DIR = "ig_yolo_dataset"
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)

for split in ["train", "val"]:
    os.makedirs(f"{DATASET_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{DATASET_DIR}/labels/{split}", exist_ok=True)

loaded_photos = [Image.open(p).convert("RGB") for p in photo_paths]

def get_random_photo(w, h):
    """ดึงภาพสุ่ม พร้อมสุ่มแต่งโทนสี (สีปกติ, ขาวดำ, มืดดาร์ก, สว่างคลีน)"""
    mode = random.random()
    if loaded_photos and mode < 0.78:
        base = random.choice(loaded_photos).copy()
        base = base.resize((w, h), Image.Resampling.BILINEAR)
        sub = random.random()
        if sub < 0.15:
            base = base.convert("L").convert("RGB")
        elif sub < 0.30:
            base = ImageEnhance.Brightness(base).enhance(random.uniform(0.25, 0.55))
        elif sub < 0.45:
            base = ImageEnhance.Brightness(base).enhance(random.uniform(1.15, 1.4))
        return base
    else:
        bg_color = tuple(random.randint(20, 240) for _ in range(3))
        img = Image.new("RGB", (w, h), bg_color)
        dr = ImageDraw.Draw(img)
        for _ in range(random.randint(3, 8)):
            c = tuple(random.randint(0, 255) for _ in range(3))
            x1, y1 = random.randint(0, max(1, w - 10)), random.randint(0, max(1, h - 10))
            x2, y2 = random.randint(x1 + 5, max(x1 + 6, w)), random.randint(y1 + 5, max(y1 + 6, h))
            if random.random() < 0.5:
                dr.rectangle([x1, y1, x2, y2], fill=c)
            else:
                dr.ellipse([x1, y1, x2, y2], fill=c)
        return img

def to_yolo_box(xmin, ymin, xmax, ymax, img_w, img_h):
    xmin = max(0, min(img_w, xmin))
    xmax = max(0, min(img_w, xmax))
    ymin = max(0, min(img_h, ymin))
    ymax = max(0, min(img_h, ymax))
    bw = xmax - xmin
    bh = ymax - ymin
    if bw <= 4 or bh <= 4:
        return None
    cx = (xmin + bw / 2.0) / img_w
    cy = (ymin + bh / 2.0) / img_h
    nw = bw / img_w
    nh = bh / img_h
    return f"{cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}"

def draw_tile_overlay(dr, x, y, w, h):
    ov = random.random()
    if ov < 0.35:
        ix, iy = x + w - 22, y + 8
        dr.rounded_rectangle([ix, iy, ix + 14, iy + 14], radius=3, outline=(255, 255, 255), width=2)
    elif ov < 0.50:
        ix, iy = x + w - 22, y + 8
        dr.polygon([(ix, iy), (ix + 14, iy + 7), (ix, iy + 14)], fill=(255, 255, 255))

def generate_negative_sample(img_w, img_h):
    neg_type = random.choice(["slip", "chat", "full_photo", "random_ui"])
    if neg_type == "full_photo":
        return get_random_photo(img_w, img_h), []
    elif neg_type == "slip":
        bg = random.choice([(245, 248, 252), (252, 250, 242), (235, 245, 240)])
        img = Image.new("RGB", (img_w, img_h), bg)
        dr = ImageDraw.Draw(img)
        dr.rounded_rectangle([24, 40, img_w - 24, img_h - 40], radius=18, fill=(255, 255, 255), outline=(210, 215, 220), width=2)
        dr.ellipse([48, 68, 104, 124], fill=(40, 160, 100))
        for row in range(6):
            ry = 160 + row * 55
            dr.rectangle([48, ry, img_w // 2, ry + 14], fill=(160, 165, 170))
            dr.rectangle([img_w - 160, ry, img_w - 48, ry + 14], fill=(80, 85, 90))
        qx, qy = img_w // 2 - 55, img_h - 190
        dr.rectangle([qx, qy, qx + 110, qy + 110], outline=(40, 40, 40), width=3)
        for _ in range(30):
            bx, by = random.randint(qx + 8, qx + 92), random.randint(qy + 8, qy + 92)
            dr.rectangle([bx, by, bx + 8, by + 8], fill=(20, 20, 20))
        return img, []
    elif neg_type == "chat":
        bg = random.choice([(140, 170, 215), (20, 20, 25), (245, 245, 248)])
        img = Image.new("RGB", (img_w, img_h), bg)
        dr = ImageDraw.Draw(img)
        cy = 70
        while cy < img_h - 80:
            bh = random.randint(36, 90)
            bw = random.randint(120, int(img_w * 0.65))
            if random.random() < 0.5:
                dr.rounded_rectangle([20, cy, 20 + bw, cy + bh], radius=16, fill=(255, 255, 255))
            else:
                dr.rounded_rectangle([img_w - 20 - bw, cy, img_w - 20, cy + bh], radius=16, fill=(130, 225, 110))
            cy += bh + random.randint(14, 28)
        return img, []
    else:
        img = Image.new("RGB", (img_w, img_h), (250, 250, 250))
        dr = ImageDraw.Draw(img)
        for y in range(40, img_h - 40, 32):
            dr.rectangle([30, y, random.randint(150, img_w - 30), y + 12], fill=(180, 180, 185))
        return img, []

def generate_ig_sample(idx):
    is_tablet = random.random() < 0.20
    img_w = random.randint(580, 760) if is_tablet else random.randint(390, 520)
    img_h = random.randint(780, 1080)

    if random.random() < 0.18:
        return generate_negative_sample(img_w, img_h)

    is_dark = random.random() < 0.55
    bg_color = random.choice([(0, 0, 0), (12, 15, 20), (18, 18, 18)]) if is_dark else random.choice([(255, 255, 255), (250, 250, 250)])
    fg_color = (245, 245, 245) if is_dark else (25, 25, 25)
    muted_color = (140, 140, 145) if is_dark else (130, 130, 135)
    btn_bg = (38, 38, 42) if is_dark else (238, 238, 240)
    line_color = (45, 45, 50) if is_dark else (220, 220, 225)

    img = Image.new("RGB", (img_w, img_h), (0, 0, 0) if is_tablet else bg_color)
    dr = ImageDraw.Draw(img)
    labels = []

    margin_x = random.randint(65, 120) if is_tablet else 0
    content_w = img_w - 2 * margin_x
    if is_tablet:
        dr.rectangle([margin_x, 0, margin_x + content_w, img_h], fill=bg_color)

    mode_roll = random.random()
    is_scrolled_only = mode_roll < 0.12
    is_zero_or_private = (not is_scrolled_only) and (mode_roll < 0.26)

    curr_y = random.randint(28, 56)

    if not is_scrolled_only:
        dr.rectangle([margin_x + 45, curr_y, margin_x + 190, curr_y + 16], fill=fg_color)
        curr_y += random.randint(34, 48)

        # 1. วาด Avatar (Class 1)
        av_size = int(content_w * random.uniform(0.18, 0.23))
        av_x = margin_x + int(content_w * 0.05)
        av_y = curr_y

        if random.random() < 0.5:
            dr.ellipse([av_x - 4, av_y - 4, av_x + av_size + 4, av_y + av_size + 4], outline=(225, 48, 108), width=3)

        av_photo = get_random_photo(av_size, av_size)
        mask = Image.new("L", (av_size, av_size), 0)
        ImageDraw.Draw(mask).ellipse([0, 0, av_size, av_size], fill=255)
        img.paste(av_photo, (av_x, av_y), mask)

        box_str = to_yolo_box(av_x, av_y, av_x + av_size, av_y + av_size, img_w, img_h)
        if box_str:
            labels.append(f"1 {box_str}")

        # 2. วาด Stats Bar 3 คอลัมน์ (Class 2: ig_stats_bar)
        st_x1 = av_x + av_size + int(content_w * 0.06)
        st_x2 = margin_x + content_w - int(content_w * 0.05)
        st_y1 = av_y + int(av_size * 0.18)
        st_y2 = av_y + int(av_size * 0.78)
        col_span = (st_x2 - st_x1) // 3
        for c in range(3):
            cx_start = st_x1 + c * col_span + 8
            dr.rectangle([cx_start, st_y1, cx_start + int(col_span * 0.55), st_y1 + 14], fill=fg_color)
            dr.rectangle([cx_start - 4, st_y1 + 22, cx_start + int(col_span * 0.75), st_y1 + 32], fill=muted_color)

        st_box = to_yolo_box(st_x1, st_y1 - 4, st_x2, st_y2 + 4, img_w, img_h)
        if st_box:
            labels.append(f"2 {st_box}")

        curr_y = av_y + av_size + random.randint(14, 24)

        for _ in range(random.randint(1, 4)):
            bw = random.randint(int(content_w * 0.25), int(content_w * 0.75))
            dr.rectangle([av_x, curr_y, av_x + bw, curr_y + 10], fill=fg_color)
            curr_y += 16

        curr_y += 10
        btn_w = (content_w - int(content_w * 0.1) - 8) // 2
        dr.rounded_rectangle([av_x, curr_y, av_x + btn_w, curr_y + 30], radius=8, fill=btn_bg)
        dr.rounded_rectangle([av_x + btn_w + 8, curr_y, av_x + btn_w * 2 + 8, curr_y + 30], radius=8, fill=btn_bg)
        curr_y += 44

        if random.random() < 0.45:
            hl_size = int(content_w * 0.13)
            for h_idx in range(random.randint(2, 5)):
                hx = av_x + h_idx * (hl_size + 16)
                if hx + hl_size < margin_x + content_w - 10:
                    dr.ellipse([hx, curr_y, hx + hl_size, curr_y + hl_size], outline=line_color, width=2)
            curr_y += hl_size + 22

    # 3. วาด Tab Bar (Class 3: ig_tab_bar)
    tab_h = 42
    dr.line([(margin_x, curr_y), (margin_x + content_w, curr_y)], fill=line_color, width=1)
    for t_idx in range(3):
        tx = margin_x + int((t_idx + 0.5) * (content_w / 3)) - 10
        ty = curr_y + 11
        dr.rectangle([tx, ty, tx + 20, ty + 20], outline=fg_color if t_idx == 0 else muted_color, width=2)
    dr.line([(margin_x + 20, curr_y + tab_h - 2), (margin_x + content_w // 3 - 20, curr_y + tab_h - 2)], fill=fg_color, width=2)

    tab_box = to_yolo_box(margin_x, curr_y, margin_x + content_w, curr_y + tab_h, img_w, img_h)
    if tab_box:
        labels.append(f"3 {tab_box}")

    curr_y += tab_h + 2

    # 4. วาดพื้นที่ด้านล่าง Tab Bar
    if is_zero_or_private:
        emp_y1 = curr_y + 35
        cx = margin_x + content_w // 2
        r_rad = 38
        dr.ellipse([cx - r_rad, emp_y1, cx + r_rad, emp_y1 + r_rad * 2], outline=fg_color, width=2)
        dr.rectangle([cx - 75, emp_y1 + r_rad * 2 + 18, cx + 75, emp_y1 + r_rad * 2 + 32], fill=fg_color)
        dr.rectangle([cx - 105, emp_y1 + r_rad * 2 + 42, cx + 105, emp_y1 + r_rad * 2 + 52], fill=muted_color)
        emp_box = to_yolo_box(cx - 115, emp_y1 - 6, cx + 115, emp_y1 + r_rad * 2 + 60, img_w, img_h)
        if emp_box:
            labels.append(f"4 {emp_box}")
    else:
        num_posts = random.choices([1, 2, 3, 4, 5, 6, 9], weights=[18, 18, 14, 15, 15, 12, 8])[0]
        is_portrait_grid = random.random() < 0.72
        gap = random.choice([1, 2, 3])
        col_w = (content_w - 2 * gap) // 3
        row_h = int(col_w * 1.333) if is_portrait_grid else col_w

        nav_limit_y = img_h - 48
        for p_idx in range(num_posts):
            r = p_idx // 3
            c = p_idx % 3
            tx = margin_x + c * (col_w + gap)
            ty = curr_y + r * (row_h + gap)
            if ty + 35 >= nav_limit_y:
                break
            vis_h = min(row_h, nav_limit_y - ty)
            tile_img = get_random_photo(col_w, vis_h)
            img.paste(tile_img, (tx, ty))
            draw_tile_overlay(dr, tx, ty, col_w, vis_h)

            if vis_h >= row_h * 0.45:
                t_box = to_yolo_box(tx, ty, tx + col_w, ty + vis_h, img_w, img_h)
                if t_box:
                    labels.append(f"0 {t_box}")

    nav_y = img_h - 46
    dr.rectangle([margin_x, nav_y, margin_x + content_w, img_h], fill=bg_color)
    dr.line([(margin_x, nav_y), (margin_x + content_w, nav_y)], fill=line_color, width=1)
    for b_idx in range(5):
        bx = margin_x + int((b_idx + 0.5) * (content_w / 5)) - 9
        dr.ellipse([bx, nav_y + 13, bx + 18, nav_y + 31], outline=fg_color, width=2)

    return img, labels

TOTAL_SAMPLES = 1500
TRAIN_COUNT = 1250

print(f"🎨 กำลังสร้างชุดข้อมูลเทรน {TOTAL_SAMPLES} ภาพ...")
for i in tqdm(range(TOTAL_SAMPLES)):
    split = "train" if i < TRAIN_COUNT else "val"
    img, lbs = generate_ig_sample(i)
    img.save(f"{DATASET_DIR}/images/{split}/sample_{i:04d}.jpg", quality=86)
    with open(f"{DATASET_DIR}/labels/{split}/sample_{i:04d}.txt", "w") as f:
        f.write("\n".join(lbs))

yaml_data = {
    "path": os.path.abspath(DATASET_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": {
        0: "post_tile",
        1: "avatar",
        2: "ig_stats_bar",
        3: "ig_tab_bar",
        4: "empty_or_private"
    }
}
with open(f"{DATASET_DIR}/dataset.yaml", "w") as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print("✅ สร้าง Dataset และไฟล์ dataset.yaml เสร็จสมบูรณ์!")

In [ ]:
# @title 4. เริ่มเทรนโมเดล YOLOv11-Nano (ใช้เวลา ~10-12 นาทีบน GPU)
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(
    data=f"{DATASET_DIR}/dataset.yaml",
    epochs=45,
    imgsz=640,
    batch=32,
    patience=12,
    project="igotyou_runs",
    name="ig_detector_v1",
    exist_ok=True
)
print("🎉 เทรนโมเดลเสร็จสมบูรณ์!")

In [ ]:
# @title 5. แปลงโมเดลเป็น ONNX และดาวน์โหลดลงเครื่องอัตโนมัติ
import shutil
from google.colab import files

best_pt = "igotyou_runs/ig_detector_v1/weights/best.pt"
best_model = YOLO(best_pt)

onnx_path = best_model.export(format="onnx", imgsz=640, simplify=True, opset=12)

target_file = "ig_detector.onnx"
shutil.copy(onnx_path, target_file)
size_mb = os.path.getsize(target_file) / (1024 * 1024)
print(f"✅ สร้างไฟล์ {target_file} สำเร็จ! (ขนาด: {size_mb:.2f} MB)")
print("⬇️ กำลังดาวน์โหลดไฟล์ลงเครื่องของคุณ... นำไปวางที่ public/models/ig_detector.onnx ได้เลย!")
files.download(target_file)